In [1]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.evaluation import ContextQAEvalChain, CriteriaEvalChain
from dotenv import load_dotenv

Load Clean Vector Store and Build RAG Chain

In [2]:
chroma_storage_dir = '../../../chroma_db_clean'

# Create embedding model
embeddings = OpenAIEmbeddings()

# Load the clean vector store
vectorstore = Chroma(
    persist_directory=chroma_storage_dir,
    embedding_function=embeddings
)

# Create a retriever
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

# Create LLM
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Define RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer the question using only the provided context. If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}')
])

# Helper to format retrieved documents
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# Build RAG chain
rag_chain = (
    {
        'context': retriever | format_docs,
        'question': RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print('RAG chain ready using clean vector store.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


RAG chain ready using clean vector store.


Define Example Questions

In [3]:
# Define a small list of evaluation questions
examples = [
    'What are the common crop diseases and their control methods?',
    'What are the top causes of death in Nigeria?',
    'What are the symptoms of Cassava Mosaic Disease?',
    'What is Mastitis and how is it managed in dairy animals?',
]

print('Examples defined.')

Examples defined.


Create Evaluators

In [4]:
# Create evaluator for faithfulness (answer grounded in context)
context_qa_evaluator = ContextQAEvalChain.from_llm(
    llm=llm,
    criteria='relevance'
)

# Create evaluator for answer relevance (answer addresses question)
relevance_evaluator = CriteriaEvalChain.from_llm(
    llm=llm,
    criteria = 'relevance'
)


Run Evaluation Loop

In [7]:
for q in examples:
    # Retrieve context
    docs = retriever.invoke(q)
    context = '\n\n'.join(doc.page_content for doc in docs)

    # Generate answer
    answer = rag_chain.invoke(q)

    # Evaluate faithfulness (pass an empty reference to satisfy the check)
    faithfulness_result = context_qa_evaluator.evaluate_strings(
        prediction=answer,
        input=q,
        context=context,
        reference=''   
    )

    # Evaluate relevance
    relevance_result = relevance_evaluator.evaluate_strings(
        prediction=answer,
        input=q
    )

    print(f'Question: {q}')
    print(f'Answer: {answer}')
    print(f'Faithful: {faithfulness_result["score"]}')
    print(f'Relevant: {relevance_result["score"]}')
    print('-' * 100)

Question: What are the common crop diseases and their control methods?
Answer: The common crop disease mentioned is Cassava Mosaic Disease. The affected crop is cassava, and the symptoms include yellowing and mottling of leaves, stunted growth, and reduced yield. The control methods are to use disease-free cuttings, plant resistant varieties, and remove infected plants early.
Faithful: 1
Relevant: 0
----------------------------------------------------------------------------------------------------
Question: What are the top causes of death in Nigeria?
Answer: The top causes of death in Nigeria are as follows:

- Malaria (20%)
- Lower Respiratory Infection (19%)
- HIV/AIDS (9%)
- Diarrheal Diseases (5%)
- Road Injuries (5%)
- Protein-energy malnutrition
- Cancer
- Meningitis
- Stroke
- Tuberculosis
Faithful: 0
Relevant: 1
----------------------------------------------------------------------------------------------------
Question: What are the symptoms of Cassava Mosaic Disease?
Answer